# 04 — Tuning Optuna + Modèle final + Model Registry (MLflow)

## Objectifs
1) Présenter la phase de fine-tuning (Optuna) sur LightGBM.
2) Comparer le modèle final "avant tuning" vs "après tuning" (même protocole final / même holdout).
3) Vérifier le versioning du modèle dans le Model Registry (v1 -> v2).

## Point important sur la métrique métier (business_cost)
La valeur brute de business_cost dépend du nombre d'exemples évalués.
Donc :
- On compare business_cost entre modèles dans le Notebook 03 (même protocole CV).
- On compare business_cost entre trials Optuna (même CV / même taille / même objectif).
- On compare business_cost entre modèles finaux (avant vs après tuning) uniquement si le holdout est identique.
En revanche, on évite de comparer directement des valeurs de business_cost issues de protocoles différents (CV vs holdout, ou tailles différentes).

In [1]:
from pathlib import Path
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

# Trouver la racine du projet (dossier 'src')
cwd = Path().resolve()
root = cwd
while root != root.parent and not (root / "src").exists():
    root = root.parent
if not (root / "src").exists():
    raise RuntimeError("Impossible de trouver la racine du projet (dossier 'src' introuvable).")

db_path = (root / "mlflow" / "mlflow.db").resolve()
if not db_path.exists():
    raise RuntimeError(f"DB MLflow introuvable : {db_path}")

TRACKING_URI = f"sqlite:///{db_path.as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiments:", [e.name for e in client.search_experiments()])

g:\Mon Drive\OC\Projet_6\credexp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: sqlite:///G:/Mon Drive/OC/Projet_6/credexp/mlflow/mlflow.db
Experiments: ['credexp', 'Default']


In [2]:
EXP_NAME = "credexp"
exp = mlflow.get_experiment_by_name(EXP_NAME)

if exp is None:
    exps = client.search_experiments()
    non_default = [e for e in exps if e.name != "Default"]
    if not non_default:
        raise RuntimeError("Aucune expérience non-Default trouvée.")
    exp = non_default[0]
    print("EXP_NAME introuvable, fallback ->", exp.name)

print("Using experiment:", exp.name, "| id:", exp.experiment_id)

Using experiment: credexp | id: 1


In [8]:
runs = mlflow.search_runs([exp.experiment_id], output_format="pandas")
print("Total runs:", len(runs))
display(runs[["run_id", "start_time", "status", "tags.mlflow.runName"]].head(20))

Total runs: 15


,run_id,start_time,status,tags.mlflow.runName
0,7663266b67f7497fa57b12a1791d0b1b,2026-02-21 19:53:22.112000+00:00,FINISHED,lgbm_final_v2
1,7dc3ff5229874d80b5423ddab7cc9afc,2026-02-21 19:20:28.894000+00:00,FINISHED,optuna_lgbm_cost_v1
2,5810003f16064b8f8fd43c374d08a718,2026-02-21 16:11:41.210000+00:00,FINISHED,lgbm
3,acd95381a9c0429891024d8c6a52e7d7,2026-02-21 16:06:21.462000+00:00,FINISHED,mlp_logistic
4,9de3b4972aed4684be188572b61c398e,2026-02-21 16:03:18.867000+00:00,FINISHED,mlp_relu
5,8448a62202cb479ea38a5464f99086d3,2026-02-21 16:00:47.503000+00:00,FINISHED,lr
6,5ccd9a3e8f5b483ea64393176302b087,2026-02-21 15:58:45.115000+00:00,FINISHED,dummy_stratified
7,bb81cdc8fff242d1bd38444329556f21,2026-02-21 15:56:43.308000+00:00,FINISHED,dummy_most_frequent
8,32b7d5c3cab5460eaa8ef0ad3c7b169b,2026-02-21 15:50:42.402000+00:00,FINISHED,lgbm
9,558eb1a388ac49c59efce010be44f2e3,2026-02-21 15:35:24.912000+00:00,FINISHED,mlp_logistic


## 1) Fine-tuning Optuna (LightGBM)

Le tuning Optuna a été exécuté via script (exemple) :
- uv run python scripts/tune_optuna.py --trials 30 --cv 3

On identifie les runs de tuning dans MLflow et on examine :
- la meilleure valeur obtenue (business cost)
- les hyperparamètres correspondants
- la stabilité (dispersion des trials si loggée)

In [9]:
# Heuristique: runName contient 'optuna' ou 'tune'
mask_tune = runs["tags.mlflow.runName"].fillna("").str.lower().str.contains("optuna|tune")
tune_runs = runs[mask_tune].copy().sort_values("start_time", ascending=False)

print("Tuning runs found:", len(tune_runs))
display(tune_runs[["run_id", "start_time", "tags.mlflow.runName"]].head(20))

Tuning runs found: 1


,run_id,start_time,tags.mlflow.runName
1,7dc3ff5229874d80b5423ddab7cc9afc,2026-02-21 19:20:28.894000+00:00,optuna_lgbm_cost_v1


In [10]:
# Colonnes utiles (si présentes)
cols = [
    "run_id",
    "start_time",
    "tags.mlflow.runName",
    "metrics.best_value",
    "metrics.best_business_cost",
    "metrics.business_cost_mean",
]
cols = [c for c in cols if c in tune_runs.columns]
display(tune_runs[cols].head(10))

,run_id,start_time,tags.mlflow.runName,metrics.best_business_cost,metrics.business_cost_mean
1,7dc3ff5229874d80b5423ddab7cc9afc,2026-02-21 19:20:28.894000+00:00,optuna_lgbm_cost_v1,50470.666667,NaN


In [11]:
# On prend le plus récent par défaut
if len(tune_runs) == 0:
    print("Aucun run Optuna détecté. (OK si tu n'as pas loggé Optuna dans MLflow.)")
else:
    run_id = tune_runs.iloc[0]["run_id"]
    data = runs[runs["run_id"] == run_id].iloc[0]

    metric_cols = sorted([c for c in runs.columns if c.startswith("metrics.")])
    param_cols = sorted([c for c in runs.columns if c.startswith("params.")])

    print("Selected tune run:", run_id)
    display(pd.DataFrame({"metric": metric_cols, "value": [data.get(c) for c in metric_cols]}).dropna())
    display(pd.DataFrame({"param": param_cols, "value": [data.get(c) for c in param_cols]}).dropna())

Selected tune run: 7dc3ff5229874d80b5423ddab7cc9afc


,metric,value
0,metrics.best_business_cost,50470.666667


,param,value
1,params.best_colsample_bytree,0.9005074287965014
2,params.best_learning_rate,0.02285471822344264
3,params.best_max_depth,6
4,params.best_min_child_samples,117
5,params.best_n_estimators,781
6,params.best_num_leaves,59
7,params.best_reg_alpha,1.2332763992778617
8,params.best_reg_lambda,4.120486736133096
9,params.best_subsample,0.6093786951987209
11,params.cost_fn,10.0


## 2) Modèles finaux (avant vs après tuning)

Le modèle final est entraîné sur un "dev set" puis évalué sur un holdout jamais vu.
On compare uniquement des runs finaux entre eux (protocole identique).

uv run python scripts/train_final.py --run-name lgbm_final_v1
uv run python scripts/train_final.py --run-name lgbm_final_v2

Dans ton cas :
- Le meilleur modèle est LightGBM AVEC undersampling.
- On veut comparer le modèle final "avant tuning" et "après tuning".

In [12]:
mask_final = runs["tags.mlflow.runName"].fillna("").str.lower().str.contains("final")
final_runs = runs[mask_final].copy().sort_values("start_time", ascending=False)

print("Final runs found:", len(final_runs))
display(final_runs[["run_id", "start_time", "tags.mlflow.runName"]].head(20))

Final runs found: 2


,run_id,start_time,tags.mlflow.runName
0,7663266b67f7497fa57b12a1791d0b1b,2026-02-21 19:53:22.112000+00:00,lgbm_final_v2
14,c4badde4de3a406cb0891d37daec7720,2026-02-21 15:14:04.270000+00:00,lgbm_final_v1


In [13]:
cols_final = [
    "run_id",
    "start_time",
    "tags.mlflow.runName",
    "metrics.val_business_cost",
    "metrics.val_best_threshold",
    "metrics.val_roc_auc",
    "metrics.val_pr_auc",
    "metrics.holdout_business_cost",
    "metrics.holdout_roc_auc",
    "metrics.holdout_pr_auc",
    "metrics.holdout_tn",
    "metrics.holdout_fp",
    "metrics.holdout_fn",
    "metrics.holdout_tp",
]
cols_final = [c for c in cols_final if c in final_runs.columns]

df_final = final_runs[cols_final].copy()

# Tri par coût holdout si dispo, sinon par date
if "metrics.holdout_business_cost" in df_final.columns:
    df_final = df_final.sort_values("metrics.holdout_business_cost", ascending=True)

display(df_final.head(10))

,run_id,start_time,tags.mlflow.runName,metrics.val_business_cost,metrics.val_best_threshold,metrics.val_roc_auc,metrics.val_pr_auc,metrics.holdout_business_cost,metrics.holdout_roc_auc,metrics.holdout_pr_auc,metrics.holdout_tn,metrics.holdout_fp,metrics.holdout_fn,metrics.holdout_tp
0,7663266b67f7497fa57b12a1791d0b1b,2026-02-21 19:53:22.112000+00:00,lgbm_final_v2,27075.0,0.49,0.785607,0.275197,15031.0,0.789157,0.291165,21287.0,6981.0,805.0,1678.0
14,c4badde4de3a406cb0891d37daec7720,2026-02-21 15:14:04.270000+00:00,lgbm_final_v1,27835.0,0.32,0.773490,0.258853,15255.0,0.782767,0.277805,20263.0,8005.0,725.0,1758.0


In [14]:
if len(df_final) == 0:
    raise RuntimeError("Aucun run final trouvé. Lance ton train_final puis re-run ce notebook.")

# Hypothèse: les deux meilleurs (ou deux plus récents) sont v2 puis v1, ou l'inverse.
# On prend les 2 premiers après tri coût holdout (si dispo), sinon les 2 plus récents.
candidates = df_final.head(2).copy()
display(candidates)

print("Selected finals:")
for _, r in candidates.iterrows():
    print("-", r["tags.mlflow.runName"], "| run_id:", r["run_id"])

,run_id,start_time,tags.mlflow.runName,metrics.val_business_cost,metrics.val_best_threshold,metrics.val_roc_auc,metrics.val_pr_auc,metrics.holdout_business_cost,metrics.holdout_roc_auc,metrics.holdout_pr_auc,metrics.holdout_tn,metrics.holdout_fp,metrics.holdout_fn,metrics.holdout_tp
0,7663266b67f7497fa57b12a1791d0b1b,2026-02-21 19:53:22.112000+00:00,lgbm_final_v2,27075.0,0.49,0.785607,0.275197,15031.0,0.789157,0.291165,21287.0,6981.0,805.0,1678.0
14,c4badde4de3a406cb0891d37daec7720,2026-02-21 15:14:04.270000+00:00,lgbm_final_v1,27835.0,0.32,0.773490,0.258853,15255.0,0.782767,0.277805,20263.0,8005.0,725.0,1758.0


Selected finals:
- lgbm_final_v2 | run_id: 7663266b67f7497fa57b12a1791d0b1b
- lgbm_final_v1 | run_id: c4badde4de3a406cb0891d37daec7720


### Conclusion (modèle final)

On retient le modèle final qui minimise le coût métier sur holdout (à protocole identique),
tout en conservant des métriques ROC-AUC et PR-AUC cohérentes.

Le modèle retenu est : LightGBM avec undersampling, version tuned.

## 3) Model Registry (preuve de versioning)

On vérifie que le modèle final est bien enregistré dans le Model Registry MLflow :
- même nom de modèle
- versions distinctes (v1 baseline, v2 tuned)

In [19]:
import mlflow
from mlflow.tracking import MlflowClient

# IMPORTANT: utilise le même tracking URI déjà vérifié plus haut
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)  # registry = même DB sqlite

client = MlflowClient()

models = client.search_registered_models()
print("Registered models:", [m.name for m in models])

# Si tu connais le nom exact dans l'UI, mets-le ici:
# Sinon, affiche d'abord la liste.

Registered models: ['credit_scoring_model']


In [21]:
MODEL_NAME = "credit_scoring_model"

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if len(versions) == 0:
    raise RuntimeError(
        f"Aucune version trouvée pour '{MODEL_NAME}'. "
        "Vérifie que train_final loggue bien registered_model_name."
    )

# Tri par version numérique
versions_sorted = sorted(versions, key=lambda v: int(v.version))

for v in versions_sorted:
    print(f"v{v.version} | run_id={v.run_id} | status={v.status} | stage={getattr(v, 'current_stage', '')}")

v1 | run_id=c4badde4de3a406cb0891d37daec7720 | status=READY | stage=None
v2 | run_id=7663266b67f7497fa57b12a1791d0b1b | status=READY | stage=None


## Synthèse finale

- Notebook 03 : sélection du meilleur modèle via CV (LightGBM + undersampling).
- Notebook 04 : tuning Optuna puis entraînement final et comparaison avant/après tuning.
- Le modèle final retenu est enregistré dans le Model Registry (MLflow), prêt pour :
  - explainability (importance + SHAP)
  - déploiement (Partie 2)